In [1]:
import json
import os, sys
import pickle
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import metrics
import seaborn as sns
import ROOT as rt


## user define
sys.path.append(".")
sys.path.append("./Utils")
from Plotter import HistStack
from CommonTrain import *
from Model import Seq

2023-03-31 22:25:10.552828: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6cb32/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-90caf/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorfl

Welcome to JupyROOT 6.26/08


# Set config

In [2]:
setting={}
with open('config/Settrain.json') as json_file:
    data = json.load(json_file)
    setting=data
INPUTDIR = setting["INPUTDIR"]
VERSION = setting["VERSION"]
# SIGNAL_MASSES = ["Zp-1700_CH-345", "Zp-2900_CH-345", "Zp-4100_CH-345", "Zp-2900_CH-1095", "Zp-4100_CH-1095", "Zp-4100_CH-1845"]
CHANNEL="ee"
# ERA="ee17"
print(VERSION)
INPUTDIR = sys.argv[1]
REGION = sys.argv[2]

# # for dev
# INPUTDIR = setting["INPUTDIR"]+'/ee18/'
# REGION = 'SR'

2803_equalpropv1dev


# Load data

In [3]:
if REGION == 'SR':
    file = 'nested_dic_df.pkl'
if REGION == 'VR':   
    file = 'nested_dic_df_PT.pkl'
# file = "/"+ERA+"/nested_dic_df.pkl"
with open(INPUTDIR+'/'+file, 'rb') as f:
    nested_dic_df = pickle.load(f)
 
dic_df=nested_dic_df['raw']
dic_df_met_u_unsort=nested_dic_df['met_u']
dic_df_met_d_unsort=nested_dic_df['met_d'] 

selected_columns = [
#                     'MC1C1_pre', 'H3PP_pre', 'H3TPP_pre', 
                    'pt_lead_pre', 'mass_pre', # by auc rank
                    'MT_pre', 'pt_trail_pre', 'Zpt_pre', 'dRll_pre', 'PuppiMET_pre',
                    'dPhilMET_pre', 'dPhillMET_pre', 'U_pre', 'dPhill_pre', 'MT2_pre',
                    'JZB_pre'] # order used in training

# # dic_df_meta={}
# # check b jet
# query_str="nbjet_pre==0"

# print("####################################################")
# print("####   (unweighted) #events after preselection  ####")
# print("####################################################")
# for key in dic_df:  
# #     dic_df_meta[key]=dic_df[key].query(query_str)
#     dic_df[key]=dic_df[key].query(query_str) #temp!
#     print(key, ":", dic_df[key].shape[0])
    
dic_df_train={}    
dic_df_met_u={}
dic_df_met_d={}
for key in dic_df:
    print(key)
    #dic_df_train[key]=pd.concat([dic_df[key][selected_columns], dic_df1[key][selected_columns]]) #when use ll channel
    #dic_df[key]=pd.concat([dic_df[key], dic_df1[key]])
#     dic_df_train[key]=dic_df_meta[key][selected_columns] # check b jet   
    dic_df_train[key]=dic_df[key][selected_columns]    
    dic_df_met_u[key]=dic_df_met_u_unsort[key][selected_columns]    
    dic_df_met_d[key]=dic_df_met_d_unsort[key][selected_columns]

Zp-4100_CH-1345
DY1500To2000
Zp-1700_CH-345
Zp-2500_CH-345
DY500To700
Zp-1700_CH-845
Zp-4100_CH-595
Zp-3300_CH-345
Zp-1700_CH-595
Zp-3700_CH-345
Zp-2100_CH-595
Zp-2900_CH-345
Zp-4100_CH-1095
Zp-2900_CH-845
Zp-3700_CH-1345
Zp-3300_CH-1595
Zp-2500_CH-595
Zp-4100_CH-345
Zp-3700_CH-1845
Zp-2500_CH-845
Zp-4100_CH-1595
Zp-4100_CH-1845
Zp-3300_CH-595
Zp-2100_CH-345
Zp-2900_CH-595
Zp-4100_CH-845
Zp-2100_CH-845
Zp-3300_CH-1345
Zp-3700_CH-1595
DY2000To3000
Zp-3700_CH-595
DY700To800
Zp-2900_CH-1345
Zp-3700_CH-1095
DY800To1000
DY1000To1500
Zp-3700_CH-845
Zp-3300_CH-1095
Zp-2900_CH-1095
TT
TTSemi
TTZ
DY
top
antitop
ZZ
WZ
WW
Data


In [4]:
print(dic_df["Zp-2500_CH-345"].columns)
print(dic_df_train["Zp-2500_CH-345"].columns)
print(dic_df_met_u["Zp-2500_CH-345"].columns)

Index(['PuppiMET_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'MET_pre', 'MET_phi_pre',
       'PuppiMET_phi_pre', 'eta_lead_pre', 'eta_trail_pre', 'phi_lead_pre',
       'phi_trail_pre', 'njet_pre', 'nbjetflav_pre', 'genweight_u_pre',
       'genweight_d_pre', 'genweight_pre', 'puweight_pre', 'puweight_u_pre',
       'puweight_d_pre', 'l1prefiring_pre', 'l1prefiring_u_pre',
       'l1prefiring_d_pre', 'effSF_pre', 'effSF_u_pre', 'effSF_d_pre',
       'topSF_pre', 'topSF_u_pre', 'topSF_d_pre', 'btagSF_pre', 'weight',
       'wopu', 'wol1', 'woeff', 'wotop', 'wobsf', 'weightpuweightUp',
       'weightpuweightDown', 'weightl1prefiringUp', 'weightl1prefiringDown',
       'weighteffSFUp', 'weighteffSFDown', 'weighttopSFUp', 'weighttopSFDown',
       'weightgenweightUp', 'weightgenweightDown'],
      dtype='object')
Index(['pt_lead_pre', 'mass_pre', 'MT_pre', 'pt_

In [5]:
def get_x(sig_key, dic_df_train):
    # x: sig and all bkgs concatenated
    x = dic_df_train[sig_key].copy(deep=True)
    for k in dic_df_train:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_df_train[k]], axis=0)
    
    return x

In [6]:
def get_xy(sig_key, dic_predict_sort):
    # x: sig and all bkgs concatenated
    x = dic_predict_sort[sig_key].copy(deep=True)
    for k in dic_predict_sort:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_predict_sort[k]], axis=0)
            
    tot_sig = dic_predict_sort[sig_key].shape[0]
    tot_bkgs = x.shape[0] - tot_sig        
    y_sig = np.full((tot_sig,1), 1)
    y_bkgs = np.full((tot_bkgs,1), 0) 
    y = np.concatenate((y_sig, y_bkgs), axis=0) 
    
    return x, y

In [7]:
import math
sys_list = ["puweight", "l1prefiring", "effSF", "topSF"]
# sys_list = ["l1prefiring"]
if REGION == 'SR':
    threshold = 0.3
else:
    threshold = 1    
    
def draw_with_systematics(model_mass, dic_df_raw, sys_list, variable, r, region):
    dic_sys = {}
    dic_sys_sqr = {}
    # nominal weight plot
    file_prefix = "Outputs/"+REGION
    hist=HistStack(dic_df_raw, variable, 'weight', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/norm_dnnscore.pdf", blind_from=threshold)     
    dic_yields = hist.get_yields_by_bin()
    bkg_norm = dic_yields['bkg']
    hist.create_root_file(file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    
    for s in sys_list:
        # up
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Up', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systUp_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysUp = dic_yields['bkg']
        dic_sys[s+'Up'] = np.array(sysUp) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        # down
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Down', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systDown_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysDown = dic_yields['bkg']
        dic_sys[s+'Down'] = np.array(sysDown) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        
    # si^2
    for s in sys_list:
        # by source
        sqrUp = [x*x if x>0 else -x*x for x in dic_sys[s+'Up']]
        sqrUp_up = [x if x>0 else 0 for x in sqrUp]
        sqrUp_down = [x if x<0 else 0 for x in sqrUp]
        sqrDown = [x*x if x>0 else -x*x for x in dic_sys[s+'Down']]
        sqrDown_up = [x if x>0 else 0 for x in sqrDown]
        sqrDown_down = [x if x<0 else 0 for x in sqrDown]
        
        sqr_up = np.array(sqrUp_up) + np.array(sqrDown_up)
        sqr_down = np.array(sqrUp_down) + np.array(sqrDown_down)
        
        dic_sys_sqr[s+'Up'] = sqr_up
        dic_sys_sqr[s+'Down'] = np.fabs(sqr_down) #store absolute
        
    print(dic_sys)
    print(dic_sys_sqr)
    # sqrt(sumi si^2)
    for i, s in enumerate(sys_list):
        if i==0:
            sum_up = dic_sys_sqr[s+'Up'].copy() # be carefull not to use shallow copy
            sum_down = dic_sys_sqr[s+'Down'].copy()
        else:
            sum_up += dic_sys_sqr[s+'Up']
            sum_down += dic_sys_sqr[s+'Down']

    dic_sys_sqr['sum_up'] = np.sqrt(sum_up)
#     print(sum_up)
#     print(sum_down)
    dic_sys_sqr['sum_down'] = np.sqrt(sum_down)
    print(dic_sys_sqr)
            
    return dic_sys_sqr

In [8]:
from multiprocessing import Pool
def wrapper_mp(model_mass):
    
#     # dump roc
#     dic_roc = {}
#     x, y = get_xy(model_mass, dic_predict_sort)
#     tpr, fpr, auc = dump_roc(model_mass, y, x, dic_predict_sort)
#     dic_roc['tpr'] = tpr
#     dic_roc['fpr'] = fpr
#     dic_roc['auc'] = auc
#     dic_nested_roc[model_mass] = dic_roc            


    #-- implement systematics --#   
    os.system("mkdir -p Outputs/"+REGION)
    os.system("mkdir -p Outputs/"+REGION+"/sys")
    sys_list = ["puweight", "l1prefiring", "effSF", "topSF"]
    # sys_list = ["puweight"]

    r = [nbin, 0, 1, "DNN score"]

    if REGION == 'SR':
        threshold = 0.3
    else:
        threshold = 1    
    
    # draw plot with syst
    dic_sys_sqr = draw_with_systematics(model_mass, dic_predict_sort, sys_list, "score"+model_mass, r, "region_dummy")

    hist=HistStack(dic_predict_sort, "score"+model_mass, 'weight', nbin, 0, 1, "DNN score", dic_sys=dic_sys_sqr) 
    
    c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/"+REGION+"/"+model_mass, blind_from=threshold)     
    hist.get_yeilds()
    hist.draw_pie(file_name="Outputs/"+REGION+"/"+model_mass)     
     
    return None

# Load model and evaluate

In [10]:
import time

nbin=20
# VERSION = '0903'
# VERSION = setting["VERSION"]
print(time.time())

one_mass = 'Zp-2500_CH-345'

if 'mm' in INPUTDIR:
    CHANNEL = 'mm'
if 'ee' in INPUTDIR:
    CHANNEL = 'ee'
if 'em' in INPUTDIR:
    CHANNEL = 'em'
    
OUTDIR = 'Outputs/'+CHANNEL
file_postfix = 'PNN_'+VERSION
# file_postfix = one_mass+VERSION

# model_masses = ['Zp-1700_CH-345', 'Zp-2500_CH-345', 'Zp-2900_CH-345', 'Zp-3300_CH-345', 'Zp-4100_CH-345']
model_masses = [m for m in dic_df_train if 'Zp' in m] # if want to run all
# model_masses = ['Zp-2500_CH-845']

# %matplotlib inline
rt.gROOT.SetBatch(rt.kTRUE)

dic_nested_roc = {}

ERA = 'ee17'

model = keras.models.load_model(OUTDIR+'/model_'+file_postfix)
#     model = keras.models.load_model(OUTDIR+'/model_binary_'+file_postfix)

Transformer = pickle.load(open(OUTDIR+'/scaler_'+file_postfix+'.pkl','rb'))

nested_dic_predict, nested_dic_predict_met_u, nested_dic_predict_met_d = [{} for _ in range(3)]

# masses loop
for model_mass in model_masses:
    masses = model_mass.split('_')
    zp_m = masses[0].split('-')
    zp_m = zp_m[1]
    ch_m = masses[1].split('-')
    ch_m = ch_m[1]
    
#     file_postfix = 'test'

    dic_df_scale, dic_df_scale_met_u, dic_df_scale_met_d = [{} for _ in range(3)]        

    for key2 in dic_df_train:
        dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
        dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
        dic_df_met_u[key2]['zp_m'] = np.full(dic_df_met_u[key2].shape[0], int(zp_m))
        dic_df_met_u[key2]['ch_m'] = np.full(dic_df_met_u[key2].shape[0], int(ch_m))
        dic_df_met_d[key2]['zp_m'] = np.full(dic_df_met_d[key2].shape[0], int(zp_m))
        dic_df_met_d[key2]['ch_m'] = np.full(dic_df_met_d[key2].shape[0], int(ch_m))
        dic_df_scale[key2] = Transformer.transform(dic_df_train[key2]) 
        dic_df_scale_met_u[key2] = Transformer.transform(dic_df_met_u[key2]) 
        dic_df_scale_met_d[key2] = Transformer.transform(dic_df_met_d[key2]) 

    # predict
    dic_predict, dic_predict_met_u, dic_predict_met_d = [{} for _ in range(3)]

    for key3 in dic_df_scale:
    #             if "Zp-" in key3 and not key in key3: # use same scaling used in model training for all masses
    #                 continue
        dic_predict[key3] = model.predict(dic_df_scale[key3], batch_size=dic_df_scale[key3].shape[0])
        dic_predict_met_u[key3] = model.predict(dic_df_scale_met_u[key3], batch_size=dic_df_scale_met_u[key3].shape[0])
        dic_predict_met_d[key3] = model.predict(dic_df_scale_met_d[key3], batch_size=dic_df_scale_met_d[key3].shape[0])
    
    nested_dic_predict[model_mass] = dic_predict
    nested_dic_predict_met_u[model_mass] = dic_predict_met_u
    nested_dic_predict_met_d[model_mass] = dic_predict_met_d
    

# array in y-axis to dataframe
dic_predict_sort, dic_predict_met_u_sort, dic_predict_met_d_sort = [{} for _ in range(3)]

for key4 in dic_df:
    new_df = dic_df[key4].copy(deep=True).reset_index(drop="True")
    new_df_met_u = pd.DataFrame()
    new_df_met_d = pd.DataFrame()
    new_df_met_u["weightmetUp"] = dic_df_met_u_unsort[key4]["weight"].copy(deep=True).reset_index(drop="True")
    new_df_met_d["weightmetDown"] = dic_df_met_d_unsort[key4]["weight"].copy(deep=True).reset_index(drop="True")    
    dic_predict_sort[key4] = new_df
    dic_predict_met_u_sort[key4] = new_df_met_u
    dic_predict_met_d_sort[key4] = new_df_met_d
    
for model_mass in model_masses:
    for key5 in dic_df:
        dic_predict_sort[key5]["score"+model_mass] = nested_dic_predict[model_mass][key5][:,0]
        dic_predict_met_u_sort[key5]["score"+model_mass] = nested_dic_predict_met_u[model_mass][key5][:,0]
        dic_predict_met_d_sort[key5]["score"+model_mass] = nested_dic_predict_met_d[model_mass][key5][:,0]
    
        
with Pool() as p:
    p.map(wrapper_mp, model_masses)  


with open(OUTDIR+'/nested_dic_roc_'+file_postfix+'.pkl', 'wb') as f:
    pickle.dump(dic_nested_roc, f)            
        


1680294860.3962333


/tmp/ipykernel_335754/3919086981.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_335754/3919086981.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_335754/3919086981.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

/tmp/ipykernel_335754/3919086981.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_335754/3919086981.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_335754/3919086981.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

/tmp/ipykernel_335754/3919086981.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_335754/3919086981.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_335754/3919086981.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

{'puweightUp': array([ 2.89725680e+01,  1.01657709e+01, -8.51224733e-01,  5.57724094e+00,
        4.33347815e-01,  1.47340235e+00, -4.06802867e+00, -1.64898617e+00,
       -4.55935791e-01,  1.57616609e+00,  5.14348803e-01,  5.46067502e-01,
        2.28182246e-01,  1.22855902e+00,  7.70558249e-01,  2.23237130e-02,
       -3.95163525e-01, -1.02537365e-01,  1.08462375e-01, -9.51899871e-01]), 'puweightDown': array([-2.86676723e+01, -9.75134432e+00,  1.45081897e-01, -3.50913341e+00,
       -6.31545484e-01, -1.53428507e+00,  2.51908888e+00,  1.28137305e+00,
        2.42739276e-01, -1.61736623e+00, -9.14354654e-01, -1.06803033e+00,
        1.31897229e-01, -9.05760620e-01, -1.36776482e-01, -2.14842301e-02,
        5.43330309e-01,  2.85085521e-01, -1.36851182e-01,  4.67985557e-01]), 'l1prefiringUp': array([0.48318825, 0.136287  , 0.06275023, 0.04010957, 0.0253449 ,
       0.01848061, 0.01561721, 0.016537  , 0.01519102, 0.00988197,
       0.00972024, 0.01013727, 0.00577866, 0.0064268 , 0.0066668

         3.57534224,   4.01926912,   3.03122182,   3.83486054])}
array('f', [16.376644134521484, 17.94869041442871, 26.768770217895508, 425.4893798828125, 964.8348999023438, 13.330436706542969, 9084.0693359375, 414.8154602050781, 19.586641311645508])
test


Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-2500_CH-845breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-2500_CH-845.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systUp_puweight.pdf_scoreZp-2500_CH-845breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systUp_puweight.pdf_scoreZp-2500_CH-845.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systDown_puweight.pdf_scoreZp-2500_CH-845breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systDown_puweight.pdf_scoreZp-2500_CH-845.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systUp_l1prefiring.pdf_scoreZp-2500_CH-845breakdown.pdf